# Cycle 2 — Preprocessing: Wyscout England Events

**Project:** Football Predictor  
**Depends on:** `cycle2_exploration_wyscout.ipynb`  
**Output:** `data/processed/wyscout_shots_processed.csv`

---

## Purpose

Transform the raw Wyscout JSON event data into a clean, model-ready CSV file containing only shot events with engineered features. No model is trained here — this notebook only prepares the data.

## Steps in This Notebook

1. Load raw events and filter to shots only
2. Extract target variable (Goal from tag 101)
3. Extract X, Y coordinates
4. Compute Distance to goal
5. Compute Angle to goal
6. Extract foot used (Left, Right, Header)
7. Encode match period
8. Merge player rank
9. Handle missing values
10. Final validation and save

## Comparison with FinalYearProject

FYP preprocessing loaded all 5 leagues and included goal position tags as features (leakage). FootballPredictor uses England only and excludes all post-shot tags. FYP also did not compute distance/angle as separate steps — they were embedded in a large combined cell.

---
## Step 1 — Load and Filter to Shots

**What it does:** Loads the England event data and keeps only open-play shot events.

**Why filter to shots only?** The model predicts whether a shot results in a goal. Non-shot events (passes, duels, fouls) are irrelevant to this task.

In [ ]:
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load raw events
with open('../data/raw/events_England.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'Raw events: {len(df):,}')

# Filter to open-play shots only
shots = df[df['subEventName'] == 'Shot'].copy().reset_index(drop=True)
print(f'After filtering to shots: {len(shots):,}')
print(f'Dropped: {len(df) - len(shots):,} non-shot events')

### Expected Output
```
Raw events: 643,150
After filtering to shots: 8,451
Dropped: 634,699 non-shot events
```

### Observations
- 8,451 shots remain — this is our working dataset for Cycle 2
- **Note on scope**: We use open-play shots only (subEventName='Shot'). Free kicks, penalties, and headers from set pieces have different dynamics and would ideally have their own xG models. Keeping them separate is standard practice in professional football analytics.

---
## Step 2 — Extract Target Variable

**What it does:** Creates the binary Goal column from the tags list.

**Why tag 101?** Tag ID 101 means 'Goal' in the Wyscout tagging system. Any shot with this tag in its list resulted in a goal.

In [ ]:
shots['Goal'] = shots['tags'].apply(lambda tags: int(any(t['id'] == 101 for t in tags)))

counts = shots['Goal'].value_counts()
print('Target distribution:')
print(f'  No Goal (0): {counts[0]:,} ({counts[0]/len(shots)*100:.1f}%)')
print(f'  Goal    (1): {counts[1]:,} ({counts[1]/len(shots)*100:.1f}%)')

### Expected Output
```
No Goal (0): ~7,500 (~89%)
Goal    (1):   ~950 (~11%)
```

### Observations
- ~10% goal rate — consistent with real Premier League statistics (~10-12% of shots result in goals)
- This class imbalance will be addressed in modelling using `scale_pos_weight` and `class_weight='balanced'`

---
## Step 3 — Extract X, Y Coordinates

**What it does:** Extracts the shot origin X and Y coordinates from the nested positions list.

**Why the first position?** The positions list contains [start_position, end_position]. The start position (index 0) is where the shot was taken from — the feature we need. The end position (where the ball ended up) is post-shot information.

In [ ]:
shots['X'] = shots['positions'].apply(lambda p: p[0]['x'] if p else np.nan)
shots['Y'] = shots['positions'].apply(lambda p: p[0]['y'] if p else np.nan)

print('X range:', shots['X'].min(), '-', shots['X'].max())
print('Y range:', shots['Y'].min(), '-', shots['Y'].max())
print('Nulls in X:', shots['X'].isnull().sum())
print('Nulls in Y:', shots['Y'].isnull().sum())
print()

# Drop rows with missing coordinates (cannot compute distance/angle without them)
before = len(shots)
shots = shots.dropna(subset=['X', 'Y']).reset_index(drop=True)
print(f'Dropped {before - len(shots)} rows with missing coordinates')
print(f'Remaining: {len(shots):,}')

### Observations
- Coordinates are on a 0-100 percentage scale of pitch dimensions
- The attacking goal is at X=100, Y=50 (centre)
- Very few or no rows should be dropped — coordinates are almost always present in Wyscout data

---
## Step 4 — Compute Distance to Goal

**What it does:** Converts percentage coordinates to metres and computes Euclidean distance from the shot position to the centre of the goal.

**Why metres?** Distance in metres is interpretable and consistent with published xG literature. Standard pitch dimensions: 105m x 68m.

**Why distance?** Distance is the single strongest predictor in any xG model — the further from goal, the harder to score.

In [ ]:
# Standard pitch dimensions
PITCH_LENGTH = 105  # metres
PITCH_WIDTH  = 68   # metres

# Convert percentage coordinates to metres
shots['X_m'] = shots['X'] / 100 * PITCH_LENGTH
shots['Y_m'] = shots['Y'] / 100 * PITCH_WIDTH

# Goal centre is at (105, 34) in metres — end of pitch, centre of width
GOAL_X = PITCH_LENGTH   # 105
GOAL_Y = PITCH_WIDTH / 2  # 34

shots['Distance'] = np.sqrt((shots['X_m'] - GOAL_X)**2 + (shots['Y_m'] - GOAL_Y)**2)

print('Distance to goal (metres):')
print(shots['Distance'].describe())
print()
print('Average distance for goals vs non-goals:')
print(shots.groupby('Goal')['Distance'].mean().rename({0: 'No Goal', 1: 'Goal'}))

### Expected Output (approximate)
```
Distance to goal (metres) - mean: ~18m

Average distance for goals vs non-goals:
  No Goal: ~19m
  Goal:    ~13m
```

### Observations
- Goals are scored on average from ~13m — much closer than the average shot (~18m)
- This confirms Distance is a strong predictor — consistent with xG literature
- Maximum distance of 50m+ corresponds to long-range attempts from the halfway line

---
## Step 5 — Compute Angle to Goal

**What it does:** Computes the angular width of the goal visible from the shot position, in degrees.

**Why angle?** A shot from directly in front of the goal has a wide angle (easier to score). A shot from a tight angle near the byline has a very small visible goal width (harder to score). Angle is the second most important feature in xG models after distance.

**How it is calculated:** The goal has two posts at (105, 30.34) and (105, 37.66) — 7.32m apart (standard goal width). We compute the angle subtended by both posts from the shot position.

In [ ]:
# Goal post positions in metres
GOAL_WIDTH = 7.32  # metres
POST_LEFT  = GOAL_Y - GOAL_WIDTH / 2  # 34 - 3.66 = 30.34
POST_RIGHT = GOAL_Y + GOAL_WIDTH / 2  # 34 + 3.66 = 37.66

# Vectors from shot position to each post
dx = GOAL_X - shots['X_m']  # always positive (shot is in front of goal)
dy_left  = POST_LEFT  - shots['Y_m']
dy_right = POST_RIGHT - shots['Y_m']

# Angle subtended by goal from shot position using atan2
angle_left  = np.arctan2(dy_left,  dx)
angle_right = np.arctan2(dy_right, dx)
shots['Angle'] = np.abs(np.degrees(angle_right - angle_left))

print('Angle to goal (degrees):')
print(shots['Angle'].describe())
print()
print('Average angle for goals vs non-goals:')
print(shots.groupby('Goal')['Angle'].mean().rename({0: 'No Goal', 1: 'Goal'}))

### Expected Output (approximate)
```
Angle to goal (degrees) - mean: ~20 degrees

Average angle for goals vs non-goals:
  No Goal: ~18 degrees
  Goal:    ~28 degrees
```

### Observations
- Goals are scored from wider angles (~28 degrees) — shots from the centre of the box
- Non-goals have narrower average angles — more attempts from wide positions and long range
- Angle is strongly correlated with Distance (close shots tend to have wide angles) but both are kept as they capture different aspects

---
## Step 6 — Extract Foot Used

**What it does:** Creates binary columns for left foot, right foot, and header shots.

**Why binary columns instead of one categorical?** Each shot can only have one foot tag. Binary encoding avoids the need for one-hot encoding later and is cleaner with tree-based models.

In [ ]:
shots['Left_Foot']  = shots['tags'].apply(lambda t: int(any(x['id'] == 401 for x in t)))
shots['Right_Foot'] = shots['tags'].apply(lambda t: int(any(x['id'] == 402 for x in t)))
shots['Header']     = shots['tags'].apply(lambda t: int(any(x['id'] == 403 for x in t)))

print('Foot used counts:')
print(f'  Right foot: {shots["Right_Foot"].sum():,}')
print(f'  Left foot:  {shots["Left_Foot"].sum():,}')
print(f'  Header:     {shots["Header"].sum():,}')

# Check for rows with no foot tag (unlabelled)
unlabelled = len(shots) - shots[['Right_Foot','Left_Foot','Header']].any(axis=1).sum()
print(f'  Unlabelled: {unlabelled:,}')

# Goal rate by foot type
print()
for label, col in [('Right foot', 'Right_Foot'), ('Left foot', 'Left_Foot'), ('Header', 'Header')]:
    sub = shots[shots[col] == 1]
    print(f'  Goal rate [{label}]: {sub["Goal"].mean()*100:.1f}% (n={len(sub):,})')

### Observations
- Right foot is most common (~62%) — reflects dominant foot distribution in professional football
- Headers have lower conversion — less control over direction and power
- Unlabelled shots: some shots may have no foot tag — these will have all three binary columns = 0, treated as unknown

---
## Step 7 — Encode Match Period

**What it does:** Converts matchPeriod ('1H', '2H') to a binary column (1H=1, 2H=0). Drops extra time rows.

**Why drop extra time?** Extra time (E1, E2) represents a small number of rows where fatigue and tactical context are very different from normal play. Including them would add noise.

In [ ]:
print('Match period distribution before filter:')
print(shots['matchPeriod'].value_counts())

# Keep only 1H and 2H
before = len(shots)
shots = shots[shots['matchPeriod'].isin(['1H', '2H'])].copy().reset_index(drop=True)
print(f'\nDropped {before - len(shots)} extra time rows')

# Encode: 1H=1, 2H=0
shots['First_Half'] = (shots['matchPeriod'] == '1H').astype(int)

print('First_Half distribution:')
print(shots['First_Half'].value_counts())
print(f'\nRemaining rows: {len(shots):,}')

### Observations
- Extra time rows (if any) removed — typically less than 1% of shots
- First_Half is a weak feature but included for completeness — second half has slightly higher goal rates due to fatigue and open play

---
## Step 8 — Merge Player Rank

**What it does:** Joins the playerankScore from playerank.json onto the shots dataset using matchId and playerId.

**Why both keys?** A player can appear in multiple matches. The playerank score is match-specific — the same player may have different scores in different matches based on their performance.

In [ ]:
with open('../data/raw/playerank.json') as f:
    player_rank_df = pd.DataFrame(json.load(f))

# Keep only the columns we need and drop duplicates (a player may have multiple rank entries per match)
rank_lookup = player_rank_df[['matchId','playerId','playerankScore']].drop_duplicates(
    subset=['matchId','playerId']
)

shots = shots.merge(rank_lookup, on=['matchId','playerId'], how='left')

missing = shots['playerankScore'].isnull().sum()
print(f'Shots with player rank: {len(shots) - missing:,} / {len(shots):,}')
print(f'Missing player rank:    {missing:,} ({missing/len(shots)*100:.1f}%)')
print()
print('playerankScore distribution:')
print(shots['playerankScore'].describe())

### Observations
- A proportion of shots will have missing player rank — players not in the ranking system (e.g. goalkeepers scoring, or players with insufficient data)
- Missing values will be imputed with the median in the next step
- playerankScore is continuous — it will be standardised with StandardScaler during modelling

---
## Step 9 — Handle Missing Values

**What it does:** Imputes missing playerankScore with the median. Verifies no other missing values exist.

**Why median imputation?** The median is robust to outliers — player rank has a skewed distribution, so the median is a more representative fill value than the mean. It is also the standard approach for missing values in tree-based models.

In [ ]:
median_rank = shots['playerankScore'].median()
shots['playerankScore'] = shots['playerankScore'].fillna(median_rank)
shots = shots.rename(columns={'playerankScore': 'Player_Rank'})

print(f'Imputed missing Player_Rank with median: {median_rank:.2f}')
print()

# Check all feature columns for nulls
feature_cols = ['X', 'Y', 'Distance', 'Angle', 'Left_Foot', 'Right_Foot', 'Header', 'First_Half', 'Player_Rank', 'Goal']
print('Null check on all feature columns:')
print(shots[feature_cols].isnull().sum())

### Expected Output
```
Imputed missing Player_Rank with median: XX.XX

Null check on all feature columns:
X              0
Y              0
Distance       0
Angle          0
Left_Foot      0
Right_Foot     0
Header         0
First_Half     0
Player_Rank    0
Goal           0
```

### Observations
- Zero missing values across all feature columns — dataset is model-ready
- Median imputation is applied only to Player_Rank — all other features are derived from coordinates or tags which are always present

---
## Step 10 — Final Validation and Save

**What it does:** Selects only the final feature columns, prints a summary, and saves to CSV.

**Why drop X_m and Y_m?** These are intermediate computation columns used to derive Distance and Angle. They are highly correlated with X and Y — keeping both sets would introduce redundancy. The model will use X and Y (0-100 scale) alongside Distance and Angle.

In [ ]:
import os

final_cols = ['X', 'Y', 'Distance', 'Angle', 'Left_Foot', 'Right_Foot', 'Header', 'First_Half', 'Player_Rank', 'Goal']
shots_final = shots[final_cols].copy()

print('Final dataset shape:', shots_final.shape)
print()
print('Column summary:')
print(shots_final.describe().round(2))
print()
print('Target distribution:')
counts = shots_final['Goal'].value_counts()
print(f'  No Goal (0): {counts[0]:,} ({counts[0]/len(shots_final)*100:.1f}%)')
print(f'  Goal    (1): {counts[1]:,} ({counts[1]/len(shots_final)*100:.1f}%)')

# Save
os.makedirs('../data/processed', exist_ok=True)
out_path = '../data/processed/wyscout_shots_processed.csv'
shots_final.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

### Expected Output
```
Final dataset shape: (~8,400, 10)

Target distribution:
  No Goal (0): ~7,500 (~89%)
  Goal    (1):   ~950 (~11%)

Saved: ../data/processed/wyscout_shots_processed.csv
```

### Observations
- 10 columns: 9 features + 1 target (Goal)
- The small row reduction from the full 8,451 is due to dropping extra time rows and any rows with missing coordinates
- 0 missing values confirmed — dataset is model-ready

### Comparison with FinalYearProject
| | FinalYearProject | FootballPredictor |
|--|--|--|
| Leagues | 5 (England + France + Germany + Italy + Spain) | England only |
| Shots | ~33,000+ | ~8,400 |
| Goal position tags | Included (leakage) | Excluded |
| Features | 15 (including post-shot tags) | 9 (pre-shot only) |
| Missing values | Not explicitly handled | Median imputation for Player_Rank |

---
## Preprocessing Complete

**Next step:** `cycle2_modelling.ipynb` — train baseline models (Dummy, Logistic Regression, Random Forest, XGBoost) on the processed shot data and evaluate using AUC-ROC.